In [1]:
from dataclasses import dataclass
import random
import time
import uuid
from typing import Any, Callable, Dict, List, Optional


# ==========================================
# 1. 基础组件与配置定义
# ==========================================
class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, cooldown: float = 5.0):
        self.state: str = "CLOSED"
        self.failure_count: int = 0
        self.failure_threshold: int = failure_threshold
        self.cooldown: float = cooldown
        self.opened_at: Optional[float] = None

    def can_call(self) -> bool:
        now = time.time()
        if self.state == "OPEN":
            if self.opened_at and (now - self.opened_at >= self.cooldown):
                self.state = "HALF_OPEN"
                return True
            return False
        return True

    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def record_failure(self):
        self.failure_count += 1
        if self.state == "HALF_OPEN" or self.failure_count >= self.failure_threshold:
            self.state = "OPEN"
            self.opened_at = time.time()


class Bulkhead:
    def __init__(self, capacity: int):
        self.capacity: int = capacity
        self.in_flight: int = 0

    def try_acquire(self) -> bool:
        if self.in_flight < self.capacity:
            self.in_flight += 1
            return True
        return False

    def release(self) -> None:
        if self.in_flight > 0:
            self.in_flight -= 1


@dataclass
class ToolConfig:
    name: str
    max_retries: int
    base_delay: float
    timeout: float
    idempotency_required: bool
    breaker: CircuitBreaker
    bulkhead: Bulkhead


# ==========================================
# 2. Tool 注册表 (Registry)
# ==========================================
class ToolRegistry:
    def __init__(self):
        self.configs: Dict[str, ToolConfig] = {}
        self.funcs: Dict[str, Callable] = {}

    def register(self, config: ToolConfig, func: Callable):
        self.configs[config.name] = config
        self.funcs[config.name] = func

    def get_config(self, name: str) -> ToolConfig:
        return self.configs[name]

    def get_func(self, name: str) -> Callable:
        return self.funcs[name]


# ==========================================
# 3. 多工具运行时引擎 (ToolRuntime)
# ==========================================
class ToolRuntime:
    def __init__(self, registry: ToolRegistry):
        self.registry = registry

    def execute(self, tool_name: str, args: Dict[str, Any], deadline: Optional[float] = None) -> Dict[str, Any]:
        config = self.registry.get_config(tool_name)
        tool_fn = self.registry.get_func(tool_name)

        # 全局统一生成或提取 幂等 Key，后续 Retry 严格复用
        if config.idempotency_required and "idempotency_key" not in args:
            args["idempotency_key"] = f"idempotent-{uuid.uuid4().hex[:8]}"

        if deadline is None:
            deadline = time.time() + 10.0  # 默认 10s Deadline 预算

        # ----------------------------------------------------
        # 步骤 1: Breaker Gate 检查 (在 Attempt Loop 外，仅查一次)
        # ----------------------------------------------------
        if not config.breaker.can_call():
            return {
                "status": "FAST_FAIL",
                "result": None,
                "error": f"CircuitBreaker for {tool_name} is OPEN",
            }

        retry_count = 0

        # ----------------------------------------------------
        # Attempt 循环：负责多次 Acquire/Release 与 Backoff
        # ----------------------------------------------------
        while True:
            # 步骤 2: Bulkhead Acquire
            if not config.bulkhead.try_acquire():
                return {
                    "status": "BULKHEAD_REJECTED",
                    "result": None,
                    "error": f"Bulkhead for {tool_name} is full (capacity={config.bulkhead.capacity})",
                }

            # 步骤 3: 执行 Tool (使用 try...finally 严格保障 release)
            raw_result = None
            exec_error = None
            try:
                raw_result = tool_fn(args)
            except Exception as e:
                exec_error = str(e)
            finally:
                # 步骤 4: 立即 Release 槽位！Backoff 期间绝不持有
                config.bulkhead.release()

            # 提取 API 模拟的状态码
            status_code = 200
            if exec_error:
                status_code = 500
            elif isinstance(raw_result, dict) and "status" in raw_result:
                status_code = raw_result["status"]

            # 步骤 5: Success / Failure 判定与分类
            if status_code == 200:
                config.breaker.record_success()
                return {
                    "status": "SUCCESS",
                    "result": raw_result,
                    "attempts": retry_count + 1,
                    "idempotency_key": args.get("idempotency_key"),
                }

            # 失败记录给 Breaker
            config.breaker.record_failure()

            # 不可重试的致命错误
            if status_code in (400, 401, 403, 404, 422):
                return {
                    "status": "FATAL_ERROR",
                    "result": raw_result,
                    "error": f"Fatal HTTP status: {status_code}",
                }

            # 步骤 6: Retry Policy 与 Budget 校验
            if retry_count >= config.max_retries:
                return {
                    "status": "MAX_RETRIES_EXCEEDED",
                    "result": raw_result,
                    "error": f"Reached max retries ({config.max_retries})",
                }

            # 计算 Backoff 与 Jitter
            backoff = config.base_delay * (2 ** retry_count) + random.uniform(0.0, 0.02)
            now = time.time()
            if (deadline - now) < (backoff + config.timeout):
                return {
                    "status": "DEADLINE_EXCEEDED",
                    "result": None,
                    "error": "Deadline exceeded before next retry backoff",
                }

            # 挂起等待 (此时 Bulkhead 槽位已空出供其他请求使用)
            time.sleep(backoff)
            retry_count += 1


# ==========================================
# 4. 模拟 Tool 实现 (带有状态控制)
# ==========================================
class SequenceMockTool:
    """按预设序列返回 Response 的 Mock 工具"""
    def __init__(self, sequence: List[int]):
        self.sequence = sequence
        self.called_keys: List[str] = []

    def __call__(self, args: Dict[str, Any]) -> Dict[str, Any]:
        self.called_keys.append(args.get("idempotency_key", ""))
        status = self.sequence.pop(0) if self.sequence else 200
        if status == 999:
            raise RuntimeError("Internal Tool Crash!")
        return {"status": status, "data": f"Response code {status}"}


# ==========================================
# 5. 场景验证
# ==========================================
if __name__ == "__main__":
    # 构建 Registry
    registry = ToolRegistry()

    # HR Tool: 允许重试，需要幂等
    hr_breaker = CircuitBreaker(failure_threshold=3)
    hr_bulkhead = Bulkhead(capacity=2)
    hr_config = ToolConfig(
        name="hr_tool",
        max_retries=2,
        base_delay=0.01,
        timeout=1.0,
        idempotency_required=True,
        breaker=hr_breaker,
        bulkhead=hr_bulkhead,
    )

    # Search Tool: 不做重试，无幂等要求，共享独立的 Breaker/Bulkhead
    search_breaker = CircuitBreaker(failure_threshold=2)
    search_bulkhead = Bulkhead(capacity=5)
    search_config = ToolConfig(
        name="search_tool",
        max_retries=0,
        base_delay=0.01,
        timeout=0.5,
        idempotency_required=False,
        breaker=search_breaker,
        bulkhead=search_bulkhead,
    )

    # 注册 2 个具有独立策略的 Tool
    hr_mock = SequenceMockTool(sequence=[503, 200])
    search_mock = SequenceMockTool(sequence=[200])

    registry.register(hr_config, hr_mock)
    registry.register(search_config, search_mock)

    runtime = ToolRuntime(registry)

    print("=== Scenario 1: 503 -> Retry -> 200 (校验 Bulkhead 释放与幂等 Key 一致性) ===")
    res1 = runtime.execute("hr_tool", {"emp_id": "E1001"})
    print(f"Status: {res1['status']}")
    print(f"Attempts: {res1['attempts']}")
    print(f"Idempotency Key Used: {res1['idempotency_key']}")
    print(f"Recorded Key Stream in Tool: {hr_mock.called_keys}")
    print(f"Bulkhead in_flight after call: {hr_bulkhead.in_flight} (Expected: 0)\n")

    print("=== Scenario 2: 两工具共用 Runtime，策略彼此隔离 ===")
    res_search = runtime.execute("search_tool", {"q": "langgraph"})
    print(f"Search Result: {res_search['status']}")
    print(f"Search Bulkhead in_flight: {search_bulkhead.in_flight} (Expected: 0)")
    print(f"HR Breaker state: {hr_breaker.state} | Search Breaker state: {search_breaker.state}\n")

    print("=== Scenario 3: 异常崩溃 (status=999) 时强行 Release，无 Slot Leak ===")
    crash_mock = SequenceMockTool(sequence=[999])
    crash_config = ToolConfig(
        name="crash_tool",
        max_retries=0,
        base_delay=0.01,
        timeout=1.0,
        idempotency_required=False,
        breaker=CircuitBreaker(),
        bulkhead=Bulkhead(capacity=1),
    )
    registry.register(crash_config, crash_mock)

    res_crash = runtime.execute("crash_tool", {})
    print(f"Crash Result: {res_crash['status']}")
    print(f"Crash Bulkhead in_flight: {crash_config.bulkhead.in_flight} (Expected: 0)")

=== Scenario 1: 503 -> Retry -> 200 (校验 Bulkhead 释放与幂等 Key 一致性) ===
Status: SUCCESS
Attempts: 2
Idempotency Key Used: idempotent-db82dc9e
Recorded Key Stream in Tool: ['idempotent-db82dc9e', 'idempotent-db82dc9e']
Bulkhead in_flight after call: 0 (Expected: 0)

=== Scenario 2: 两工具共用 Runtime，策略彼此隔离 ===
Search Result: SUCCESS
Search Bulkhead in_flight: 0 (Expected: 0)
HR Breaker state: CLOSED | Search Breaker state: CLOSED

=== Scenario 3: 异常崩溃 (status=999) 时强行 Release，无 Slot Leak ===
Crash Result: MAX_RETRIES_EXCEEDED
Crash Bulkhead in_flight: 0 (Expected: 0)
